# Go2 contact-aware launch and continuation feasibility
Python 3.11. Upload `contact_transition_molab_20261112_v1.zip` beside this notebook. No cell runs BC or PPO. The default replays archived splines on fresh perturbed standing resets; a new offline CEM search is a separate option.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,zipfile,subprocess
assert sys.version_info[:2]==(3,11)
root=Path("go2_contact_transition_workspace")
assert not root.exists(), "Use a fresh workspace"
with zipfile.ZipFile("contact_transition_molab_20261112_v1.zip") as z:
    z.extractall(root)
os.chdir(root)
for name,digest in json.loads(Path("contact_transition_bundle_manifest.json").read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
subprocess.run([sys.executable,"-m","pip","install","-r","contact_transition_requirements.txt"],check=True)
os.environ.update(MUJOCO_GL="egl",WANDB_MODE="disabled",MPLCONFIGDIR="/tmp/go2_contact_mpl",XDG_CACHE_HOME="/tmp/go2_contact_cache")
def run(*args):
    subprocess.run([sys.executable,"-m",*args],check=True)


In [ ]:
run("pytest","-q","experiments/recoverability/test_core.py","experiments/recoverability/test_transition.py","experiments/recoverability/bidirectional/test_bidirectional.py","experiments/recoverability/contact_transition/test_contact_transition.py")


In [ ]:
run("experiments.recoverability.contact_transition.run","--output","output/molab_contact_validation","--search-dir","output/contact_transition_search_20261112_v1","--population","24","--iterations","5","--strategies","load_yield","no_yield")


In [ ]:
run("experiments.recoverability.contact_transition.mechanics","--output","output/molab_contact_mechanics")
run("experiments.recoverability.contact_transition.extension_sweep","--output","output/molab_contact_extension")
run("experiments.recoverability.contact_transition.catch_refine","--source-candidate","output/contact_transition_catch_20261112_v1/source_candidate.json","--output","output/molab_contact_catch")


In [ ]:
run("experiments.recoverability.contact_transition.report","--search","output/contact_transition_search_20261112_v1","--validation","output/molab_contact_validation","--mechanics","output/molab_contact_mechanics","--extension","output/molab_contact_extension","--catch","output/molab_contact_catch","--output","output/molab_contact_report")
print(Path("output/molab_contact_report/measured_report.md").read_text())
print(Path("output/molab_contact_validation/reference_gate.json").read_text())


In [ ]:
RUN_NEW_OFFLINE_SEARCH=False
if RUN_NEW_OFFLINE_SEARCH:
    run("experiments.recoverability.contact_transition.run","--output","output/molab_contact_new_search","--population","24","--iterations","5","--strategies","load_yield","no_yield")


The archived experiments found no accepted repeatable reference. Training remains disabled. Higher-pitch diagnostic trajectories must not be used as valid demonstrations merely because they cross 45 degrees.